# Step 3C: human validation of classification

This notebook loads the latest Step 3B classifications and uses a separate Step 3C review workspace. It reuses the original 200-ad representative sample when suitable, retaining full descriptions, original titles and any existing human fields. No classifier is rerun and no original Step 3/3B file is edited.

Run `python -m src.detect_duplicate_jobs` from the project root to prepare the review workspace and duplicate investigation explicitly. If no matching workspace exists, this notebook prepares one once; subsequent runs reuse it. Only pandas, Jupyter/IPython and the standard library are used. These are advertisements, not independently verified unique vacancies.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'data').is_dir())
sys.path.insert(0, str(ROOT))
from src.validate_classification import (latest_step3b, read_csv, sha256, evaluate_review, save_review_edits)
from src.classify_jobs_v2 import CATEGORIES, RELEVANCES
processed = latest_step3b(ROOT / 'data' / 'processed')
source_path = processed / 'jobs_classified_v2.csv'
jobs = read_csv(source_path)
source_hash = sha256(source_path)
print('Step 3B source:', processed.name, '| advertisements:', len(jobs))
display(jobs.relevance.value_counts().rename_axis('Predicted relevance').to_frame('Advertisements'))

## Select the Step 3C review workspace

The default selects the latest Step 3C workspace tied to the exact source dataset SHA-256. Set `REVIEW_DIR` explicitly to continue an earlier review; keep using that folder so you do not abandon human work when a new run is created. Original Step 3B sample files stay untouched. This notebook does not create another redundant sample every time it runs.

In [ ]:
REVIEW_DIR = None  # Example: ROOT / 'data' / 'processed' / 'step3c-<run-id>'
if REVIEW_DIR is None:
    matches = []
    for summary_path in (ROOT / 'data' / 'processed').glob('step3c-*/step3c_summary.json'):
        meta = json.loads(summary_path.read_text(encoding='utf-8'))
        if meta['source_sha256'] == source_hash:
            matches.append(summary_path.parent)
    if not matches:
        from src.detect_duplicate_jobs import run as prepare_step3c
        prepared_dir, _ = prepare_step3c(processed=processed)
        matches.append(prepared_dir)
    review_dir = max(matches, key=lambda p: p.name)
else:
    review_dir = Path(REVIEW_DIR)
meta = json.loads((review_dir / 'step3c_summary.json').read_text(encoding='utf-8'))
if meta['source_sha256'] != source_hash:
    raise ValueError('Selected review workspace belongs to a different source dataset')
representative_path = review_dir / 'manual_review_representative.csv'
diagnostic_path = review_dir / 'manual_review_diagnostic.csv'
representative = read_csv(representative_path)
diagnostic = read_csv(diagnostic_path)
assert set(representative.advertisement_id).isdisjoint(diagnostic.advertisement_id)
source_by_id = jobs.set_index('advertisement_id')
for sample in [representative, diagnostic]:
    for col in jobs.columns:
        if col != 'advertisement_id':
            pd.testing.assert_series_equal(sample[col].reset_index(drop=True),
                source_by_id.loc[sample.advertisement_id, col].reset_index(drop=True), check_names=False)
print('Review workspace:', review_dir)
print('Original representative sample reused:', meta['review_preparation']['representative_reused'])
print('Representative rows:', len(representative), '| diagnostic rows:', len(diagnostic))

## Representative and diagnostic samples are different

The reused representative sample is a uniform random sample of 200 advertisements, seed 42; inclusion probability is 200/5,023. It includes relevant, irrelevant and uncertain predictions. If a source has no existing representative sample, the preparation module uses recorded proportional strata with at least one row from each available prediction and saves the actual inclusion probabilities.

The diagnostic sample uses seed 44 and separately targets suspected false positives, suspected false negatives and category conflicts. These are error hypotheses, not verified mistakes. Triggered relevant cases are prioritized; some relevant/irrelevant cases are included for audit even without a confirmed error signal. It excludes representative-sample IDs. Never pool the two samples to report population performance. Partial or selective human review can bias results, and duplicate postings can create correlated observations.

In [ ]:
display(representative.relevance.value_counts().rename_axis('Representative prediction').to_frame('Rows'))
display(diagnostic.diagnostic_stratum.value_counts().rename_axis('Diagnostic reason').to_frame('Rows'))
display(representative[['advertisement_id', 'job_title', 'relevance', 'primary_category',
                        'duplicate_group_id', 'duplicate_review_status']].head(15))
print('Allowed relevance labels:', RELEVANCES)
print('Allowed primary-category labels:')
print(*CATEGORIES, sep=chr(10))

## Inspect a complete advertisement

The CSVs retain every original title and full description. The function below prints full text rather than a truncated DataFrame cell. Check duplicate siblings using the full Step 3C annotated dataset when the group ID is nonempty. Similar text does not prove that two advertisements represent one vacancy. Full original classification evidence remains available in the sample.

In [ ]:
def inspect_ad(advertisement_id, sample='representative'):
    path = representative_path if sample == 'representative' else diagnostic_path
    review = read_csv(path)
    rows = review.loc[review.advertisement_id.eq(str(advertisement_id))]
    if len(rows) != 1:
        raise ValueError('Select an ID in this sample')
    row = rows.iloc[0]
    print('ID:', row.advertisement_id, '| title:', row.job_title)
    print('Prediction:', row.relevance, '| category:', row.primary_category or '(null)')
    print('Employer:', row.employer, '| date:', row.publication_date, '| location:', row.municipality)
    print('Duplicate group:', row.duplicate_group_id or '(not flagged)', '| status:', row.duplicate_review_status)
    print('FULL ORIGINAL DESCRIPTION')
    print(row.description_text)
    print('CLASSIFICATION EVIDENCE')
    print(json.dumps(json.loads(row.evidence), ensure_ascii=False, indent=2))
    if row.duplicate_group_id:
        annotated = read_csv(review_dir / 'jobs_duplicate_review.csv')
        display(annotated.loc[annotated.duplicate_group_id.eq(row.duplicate_group_id),
            ['advertisement_id', 'job_title', 'publication_date', 'municipality', 'number_of_vacancies']])

# Uncomment and choose a sample ID:
# inspect_ad(representative.advertisement_id.iloc[0])
# inspect_ad(diagnostic.advertisement_id.iloc[0], sample='diagnostic')


## Enter human labels

Read the complete advertisement and [classification policy](../docs/classification_policy.md). You can edit the Step 3C review CSV in a UTF-8 CSV editor, or enter edits in the dictionaries below. Do not edit IDs, predictions, source text or sampling metadata.

- `human_relevance`: relevant, irrelevant, or uncertain. Use uncertain if a human cannot resolve relevance.
- `human_primary_category`: an exact category above for a relevant advertisement, or empty if its category is unresolved. Always empty for irrelevant/uncertain human relevance.
- `reviewer`: identify the human who checked the advertisement.
- `human_verified`: set to `true` only after that human has checked the labels. Entered-but-unverified labels, AI predictions and assistant investigation notes are never ground truth.
- `review_notes`: record reasoning, ambiguity or suspected source problems.

After workspace preparation, leave both dictionaries empty to keep review files read-only. Saving accepts only human fields in this Step 3C folder. It never automatically sets verification or overwrites Step 3B files. A human may verify an unresolved judgment, which is reported separately and excluded from resolved-ground-truth metrics.

In [ ]:
representative_edits = {
    # 'ADVERTISEMENT_ID': {
    #     'human_relevance': 'relevant',
    #     'human_primary_category': 'Data Engineer',
    #     'reviewer': 'Your name',
    #     'review_notes': 'Reason based on actual duties',
    #     'human_verified': 'true',  # Only after a human has checked the label.
    # },
}
diagnostic_edits = {}
if representative_edits:
    save_review_edits(representative_path, representative_edits)
if diagnostic_edits:
    save_review_edits(diagnostic_path, diagnostic_edits)
print('No automatic human verification is performed.')

## Evaluation denominators and unresolved records

Only `human_verified=true` with an identified reviewer can contribute. A human uncertain judgment is unresolved, not a ground-truth class for accuracy. A verified relevant row with an empty category can contribute to relevance metrics but not primary-category metrics. Unreviewed means not explicitly human-verified; typed unverified labels are counted separately.

The corpus counts below describe this workspace only; no labels from other review files are silently imported. The entire corpus remains unreviewed until a human verifies advertisements, even if the automated review_required flag is false.

In [ ]:
representative = read_csv(representative_path)
diagnostic = read_csv(diagnostic_path)
results = {'representative': evaluate_review(representative), 'diagnostic': evaluate_review(diagnostic)}
for name, result in results.items():
    print(name.upper(), result['status'])
    display(pd.Series(result['counts'], name='Value').to_frame())
verified_ids = set()
for sample in [representative, diagnostic]:
    mask = sample.human_verified.str.strip().str.casefold().eq('true')
    verified_ids.update(sample.loc[mask, 'advertisement_id'])
print('Corpus advertisements:', len(jobs))
print('Human-verified advertisements in this workspace:', len(verified_ids))
print('Corpus advertisements not human-verified in this workspace:', len(jobs) - len(verified_ids))
print('Predicted uncertain relevance in corpus:', jobs.relevance.eq('uncertain').sum())
print('Predicted relevant with unresolved category in corpus:',
      (jobs.relevance.eq('relevant') & jobs.primary_category.eq('')).sum())

## Precision, recall, F1 and confusion matrices after verification

Relevance metrics use only resolved human relevant/irrelevant labels. Predicted uncertain remains an abstention column and reduces recall for the true class. Confusion matrices have human rows and predicted columns.

Primary-category metrics use verified human-relevant advertisements with a known human category. A prediction of irrelevant, uncertain or a null category becomes an abstention/error, so missed relevant jobs are not silently dropped. False relevant predictions on human-irrelevant advertisements are assessed in the relevance table; the primary-category table is conditional on human-relevant rows.

Undefined per-class precision/recall/F1 is null (displayed NaN). Macro averages include human-supported classes only, using zero for undefined scores on supported classes. Weighted F1 weights by human support. These are reviewed-sample metrics, not an established population or unique-vacancy accuracy. The representative and diagnostic samples are always displayed separately. No score is calculated when there is no verified, resolved human truth.

In [ ]:
for name, result in results.items():
    print('SAMPLE:', name)
    for task in ['relevance', 'primary_category']:
        print('Task:', task)
        metrics = result[task]
        if metrics is None:
            print('Metrics unavailable: no verified resolved human labels for this task.')
            continue
        display(pd.Series(metrics['summary'], name='Value').to_frame())
        display(metrics['per_class'])
        display(metrics['confusion_matrix'])

## Scope filters and duplicate limitations

Read [classification_policy.md](../docs/classification_policy.md) before comparing role counts. Core categories, AI/ML research, statisticians and management have separate role_scope values. Doctoral, internship, thesis, postdoctoral and student-work indicators are independent title flags. `core_dashboard_candidate` is a provisional scope filter; it is not a dashboard or a validated market-demand count. Relevance predictions are unchanged.

Duplicate review preserves all records. Exact text can still describe separate vacancies, reposts or multi-location advertising. Near-match graph groups can be connected without every member matching every other member. Do not remove duplicate rows from the review sample or silently switch the evaluation unit after inspecting outcomes. Any later vacancy-level analysis requires an explicit human adjudication and sampling policy.